# PTL-PINNs: Perturbation-Guided Transfer Learning with Physics-Informed Neural Networks for Nonlinear Systems

**Paper:** Alexandrino, D., Moseley, B., Protopapas, P. (2026). *PTL-PINNs: Perturbation-Guided Transfer Learning with Physics-Informed Neural Networks for Nonlinear Systems.* arXiv:2601.12093 [cs.LG].

**Carpeta origen:** `PINNs/3. Arquitecturas, frameworks y variantes/PTL-PINNs_Perturbation-Guided_Transfer_Learning_wi.pdf`

## Como se usan las PINNs en este paper

PTL-PINNs combina **teoria de perturbaciones** con **transferencia de aprendizaje de un solo paso (one-shot)** para resolver EDOs/EDPs debilmente no lineales sin descenso de gradiente en la etapa de resolucion. La idea central (Eq. 1-3): un problema debilmente no lineal $\hat D[u]+\varepsilon\mathcal{N}[u]=\mathcal{F}$ se expande en serie de potencias de $\varepsilon$:

$$u\approx u_0+\varepsilon u_1+\varepsilon^2 u_2+\dots,\qquad \mathcal{O}(1):\hat D[u_0]=\mathcal{F},\quad \mathcal{O}(\varepsilon):\hat D[u_1]=\mathcal{N}[u_0],\ \dots$$

Como resultado, el problema no lineal se convierte en una **secuencia de subproblemas lineales que comparten el mismo operador lineal** $\hat D$ y solo difieren en su termino forzante. PTL-PINNs explota esto en dos etapas:

1. **Etapa 1 (preentrenamiento, Multi-Headed-PINN)**: se entrena una red troncal compartida $H(t)$ (representacion latente) junto con multiples "cabezas" $W_k$, cada una resolviendo $\hat D[u_k]=F_k$ para distintas funciones forzantes $F_k$ de una familia representativa, via descenso de gradiente estandar.
2. **Etapa 2 (transferencia one-shot)**: como $u(t)=H(t)W$ es **lineal en los pesos de salida** $W$ y $\hat D$ es lineal, el residuo de cualquier nuevo subproblema (con la MISMA $\hat D$ pero forzante distinta) es **cuadratico en $W$**, por lo que el $W$ optimo se obtiene resolviendo un **sistema lineal en forma cerrada** (minimos cuadrados), sin iteraciones de descenso de gradiente. Esto se repite para cada orden de perturbacion, reutilizando siempre la misma $H(t)$ congelada.

El resultado es una solucion hasta $\mathcal{O}(\varepsilon^p)$ obtenida esencialmente con el costo de unas pocas multiplicaciones matriciales, hasta un orden de magnitud mas rapido que Runge-Kutta.

Este cuaderno reproduce fielmente el mecanismo completo para el **oscilador de Duffing debilmente no lineal amortiguado** ($\ddot u+2\delta\dot u+\omega_0^2u+\varepsilon u^3=0$, uno de los osciladores no lineales del benchmark del paper): preentrenamiento de una red troncal multi-cabeza, y resolucion **sin gradiente** (minimos cuadrados cerrados) de los subproblemas de orden 0 y orden 1 de la expansion perturbativa.

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio en las paginas revisadas, ni fue posible confirmar uno en esta sesion. Como referencia general del framework PINN base sobre el que se apoya:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Oscilador de Duffing debilmente no lineal: $\ddot u+2\delta\dot u+\omega_0^2u+\varepsilon u^3=0$, $u(0)=1,\dot u(0)=0$

In [ ]:
delta, omega0, eps = 0.1, 2.0, 0.3   # regimen subamortiguado, no linealidad cubica debil
t_max = 6.0

def rk4_reference(n_steps=6000):
    dt = t_max / n_steps
    state = np.array([1.0, 0.0])  # u, u_dot
    traj = [state.copy()]
    def rhs(s):
        u, v = s
        return np.array([v, -2 * delta * v - omega0**2 * u - eps * u**3])
    for _ in range(n_steps):
        k1 = rhs(state); k2 = rhs(state + 0.5*dt*k1); k3 = rhs(state + 0.5*dt*k2); k4 = rhs(state + dt*k3)
        state = state + (dt/6) * (k1 + 2*k2 + 2*k3 + k4)
        traj.append(state.copy())
    return np.array(traj)

traj_ref = rk4_reference()
t_ref = np.linspace(0, t_max, len(traj_ref))
u_ref = traj_ref[:, 0]

## 2. Etapa 1: preentrenamiento de la red troncal Multi-Headed-PINN (representacion latente compartida $H(t)$)

Se entrena una red troncal con $K$ cabezas, cada una resolviendo $\hat D[u_k]=F_k(t)$ para una forzante aleatoria distinta, con $\hat D[u]=\ddot u+2\delta\dot u+\omega_0^2 u$ (el operador lineal compartido por todos los ordenes de perturbacion del oscilador de Duffing).

In [ ]:
m_latent = 24  # ancho de la representacion latente H(t)
K_heads = 10   # numero de cabezas de preentrenamiento

class Backbone(nn.Module):
    def __init__(self, n_hidden=3, n_neurons=32, m=m_latent):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, m)]
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t)  # H(t), shape (N, m)


backbone = Backbone().to(device)
heads = nn.Parameter(torch.randn(m_latent, K_heads, device=device) * 0.1)

t_col = torch.linspace(0, t_max, 200, device=device).view(-1, 1).requires_grad_(True)
t0 = torch.zeros(1, 1, device=device, requires_grad=True)

# K forzantes aleatorias representativas (combinaciones de senos de baja frecuencia)
freqs = torch.rand(K_heads, 3, device=device) * 3
amps = (torch.rand(K_heads, 3, device=device) - 0.5) * 2

def forcing_batch(t):
    # (N,1),(K,3) -> (N,K)
    return sum(amps[:, j] * torch.sin(freqs[:, j] * t) for j in range(3))


def d_dt(f, t):
    return torch.autograd.grad(f, t, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]


opt = torch.optim.Adam(list(backbone.parameters()) + [heads], lr=2e-3)
for epoch in range(3000):
    opt.zero_grad()
    H = backbone(t_col)                 # (N, m)
    U = H @ heads                        # (N, K): todas las cabezas a la vez

    d2U_list = []
    dU_list = []
    for k in range(K_heads):
        uk = U[:, k:k+1]
        duk = d_dt(uk, t_col)
        d2uk = d_dt(duk, t_col)
        dU_list.append(duk)
        d2U_list.append(d2uk)
    dU = torch.cat(dU_list, dim=1)
    d2U = torch.cat(d2U_list, dim=1)

    F_target = forcing_batch(t_col)
    residual = d2U + 2 * delta * dU + omega0**2 * U - F_target
    loss_pde = torch.mean(residual**2)

    H0 = backbone(t0)
    U0 = H0 @ heads
    dU0 = torch.cat([d_dt(U0[:, k:k+1], t0) for k in range(K_heads)], dim=1)
    loss_ic = torch.mean(U0**2) + torch.mean(dU0**2)

    loss = loss_pde + 10.0 * loss_ic
    loss.backward()
    opt.step()
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e}')

## 3. Etapa 2: transferencia *one-shot* -- resolver los subproblemas de orden 0 y orden 1 SIN descenso de gradiente

Con $H(t)$ congelada, $u(t)=H(t)W$ es lineal en $W$, y el operador $\hat D$ tambien es lineal, por lo que $\hat D[u](t)=\hat D[H](t)\cdot W$. Se arma una unica matriz $A$ (independiente de la forzante) y se resuelve $AW=b$ por minimos cuadrados para cada orden.

In [ ]:
for p in backbone.parameters():
    p.requires_grad_(False)

t_solve = torch.linspace(1e-3, t_max, 300, device=device).view(-1, 1).requires_grad_(True)

H_solve = backbone(t_solve)                                   # (N, m)
dH_list = [d_dt(H_solve[:, j:j+1], t_solve) for j in range(m_latent)]
dH = torch.cat(dH_list, dim=1)                                 # (N, m)
d2H_list = [d_dt(dH[:, j:j+1], t_solve) for j in range(m_latent)]
d2H = torch.cat(d2H_list, dim=1)                                # (N, m)

DH = d2H + 2 * delta * dH + omega0**2 * H_solve                 # D[H](t), (N, m) -- FIJO, no depende de la forzante

t0_s = torch.zeros(1, 1, device=device, requires_grad=True)
H0_s = backbone(t0_s)
dH0_list = [d_dt(H0_s[:, j:j+1], t0_s) for j in range(m_latent)]
dH0 = torch.cat(dH0_list, dim=1)

# Matriz A: comun a TODOS los ordenes de perturbacion (solo depende de D y H, no de la forzante).
# Las dos ultimas filas (condiciones iniciales) se escalan por 1e2 para pesarlas mas que el residuo.
A = torch.cat([DH, H0_s * 1e2, dH0 * 1e2], dim=0).detach().cpu().numpy()
H_solve_np = H_solve.detach().cpu().numpy()

# Orden 0: D[u0] = 0, con u0(0)=1, u0'(0)=0 (escalado x1e2 en las filas de IC)
b0 = np.concatenate([np.zeros(t_solve.shape[0]), [1e2 * 1.0], [0.0]])
W0, *_ = np.linalg.lstsq(A, b0, rcond=None)
u0_solve = H_solve_np @ W0

# Orden 1: D[u1] = -u0^3, con u1(0)=0, u1'(0)=0
b1 = np.concatenate([-(u0_solve**3), [0.0], [0.0]])
W1, *_ = np.linalg.lstsq(A, b1, rcond=None)
u1_solve = H_solve_np @ W1

u_ptl = u0_solve + eps * u1_solve
t_solve_np = t_solve.detach().cpu().numpy().flatten()

## 4. Resultados: PTL-PINN ($u_0+\varepsilon u_1$, sin gradiente en la Etapa 2) vs. referencia RK4

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(t_ref, u_ref, label='RK4 (referencia, no lineal completo)', linewidth=2)
plt.plot(t_solve_np, u0_solve, '--', label='$u_0$ (orden 0, lineal)', alpha=0.6)
plt.plot(t_solve_np, u_ptl, label='PTL-PINN: $u_0+\\varepsilon u_1$ (one-shot, sin gradiente)')
plt.xlabel('t'); plt.ylabel('u(t)')
plt.title('Oscilador de Duffing debilmente no lineal: PTL-PINN vs RK4')
plt.legend()
plt.show()

u_ptl_interp = np.interp(t_ref, t_solve_np, u_ptl)
err = 100 * np.linalg.norm(u_ptl_interp - u_ref) / np.linalg.norm(u_ref)
print(f'Error relativo L2 de PTL-PINN (orden 0+1) vs RK4: {err:.2f}%')